# 📈 Notebook 05: Dự báo Chuỗi Thời gian & Đánh giá Tổng hợp
## Dự báo nhiệt độ + Actionable Insights


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')

import pandas as pd
from IPython.display import Image, display
from src.data.loader import load_config, resolve_path
from src.models.forecasting import TimeForecaster

config = load_config()
df = pd.read_parquet(resolve_path(config['data']['cleaned_parquet']))
print(f'✅ Loaded: {df.shape}')


## 1. Dự báo Chuỗi Thời gian


In [ ]:
forecaster = TimeForecaster(config)
results = forecaster.run(df)
forecaster.save_results()


## 2. ACF/PACF


In [ ]:
display(Image(filename=results['acf_plot']))


### Nhận xét ACF/PACF
- ACF giảm dần → tính xu hướng mạnh
- Chu kỳ ~365 ngày → xác nhận mùa vụ hàng năm
- PACF suy giảm nhanh sau lag 1-2 → AR(1)/AR(2)


## 3. So sánh Dự báo


In [ ]:
display(results['results_df'])
display(Image(filename=results['comparison_plot']))


## 4. Phân tích Phần dư


In [ ]:
residual = results['residual_analysis']
print(f"Outlier: {residual['n_outliers']}/804 ({residual['n_outliers']/804:.1%})")
display(Image(filename=residual['plot_path']))


## 5. Top ngày dự báo lệch xa


In [ ]:
if not residual['outlier_df'].empty:
    top = residual['outlier_df'].reindex(
        residual['outlier_df']['Residual'].abs().sort_values(ascending=False).index
    ).head(10)
    display(top)


---
# 🎯 ACTIONABLE INSIGHTS


In [ ]:
insights = [
    '🔹 INSIGHT 1 — CẢNH BÁO TUYẾT MÙA ĐÔNG:\n'
    '   Sương mù + Nhiệt độ thấp + Gió yếu → Tuyết 55%. Tự động cảnh báo.',
    '🔹 INSIGHT 2 — 2 CHẾ ĐỘ THỜI TIẾT:\n'
    '   Lạnh-Ẩm (47%) vs Nóng-Khô (53%). 2 profile năng lượng riêng.',
    '🔹 INSIGHT 3 — MÙA XUÂN KHÓ DỰ BÁO:\n'
    '   Lỗi 48.2% (Spring) vs 34.4% (Winter). Cần features giao mùa.',
    '🔹 INSIGHT 4 — NAIVE HIỆU QUẢ:\n'
    '   MAE=1.55°C, vượt Holt-Winters. Phù hợp dự báo ngắn hạn.',
    '🔹 INSIGHT 5 — CỰC TRỊ DỄ PHÂN LOẠI:\n'
    '   29.9% lỗi (cực trị) vs 44.6% (bình thường). Pattern rõ ràng.',
    '🔹 INSIGHT 6 — ĐỘ ẨM TĂNG KHI GIÓ YẾU MÙA HÈ:\n'
    '   Conf=65.5%. Tiết kiệm nước tưới ngày gió yếu.',
    '🔹 INSIGHT 7 — 20% NGÀY OUTLIER:\n'
    '   166/804 ngày sai >±7.77°C. Cần external features.',
]
for ins in insights:
    print(ins + '\n')


## Kết luận

1. **EDA & Tiền xử lý**: 3 bẫy xử lý, Season/Binning/Scaling
2. **Luật Kết hợp**: 80 luật FP-Growth theo 4 mùa
3. **Phân cụm**: K=2, 2 chế độ thời tiết rõ ràng
4. **Phân loại**: RF F1=0.44, error analysis chi tiết
5. **Anomaly Detection**: IF vs LOF (nhánh thay thế Criterion F)
6. **Chuỗi thời gian**: ACF/PACF + Naive vs HW + Residual
7. **7 Actionable Insights** có thể áp dụng thực tế
